# Extended Data Figure 3: Readout planes

Run from this notebook directory. The first row shows the mean long-run cooperation $\bar\rho$ over 512 area-uniform initial strategy compositions on a 41 x 41 grid of one-step readout parameters $(\alpha,\beta)$; the second row shows local invasion of a pure DISC population on an 801 x 801 grid. The cooperation plane is produced by the verified strategy-dynamics engine (`../engine/scan.py readout`), which evaluates reputations exactly wherever the all-good reputation field is not smooth, and is loaded from `data/extended_data_figure3_planes/cooperation_readout_plane.csv`. Dependencies: NumPy, pandas, and Matplotlib.

## Configuration

In [1]:
from pathlib import Path
import json
import numpy as np
from numpy.polynomial import Polynomial
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, BoundaryNorm, FuncNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ROOT = Path.cwd()
DATA = ROOT / 'data' / 'extended_data_figure3_planes'
FIGURES = ROOT / 'figures' / 'extended_data_figure3_planes'
DATA.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
RECOMPUTE_INVASION = False
MODEL = dict(b_over_c=5.0, e1=.02, e2=.02, k1=1, k2=1)
INVASION_POINTS = 801
NORMS = ('SC', 'SH', 'SJ', 'SS')
NORM_PARAMETERS = {'SC': (1., 0.), 'SH': (0., 0.), 'SJ': (0., 1.), 'SS': (1., 1.)}
NORM_NAMES = {'SC': 'Scoring', 'SH': 'Shunning', 'SJ': 'Stern Judging', 'SS': 'Simple Standing'}
RELAX_DT, RELAX_TOL, MAX_RELAX_STEPS = .18, 1e-11, 120000
INVASION_TOL = 1e-10
POLISH_TOL, POLISH_STEP = 1e-13, 1e-6   # Newton refinement of resident roots (as in Extended Data Figure 5)
FIELDS = ('r_ALLC', 'r_ALLD', 'r_DISC', 'lambda_ALLD', 'lambda_ALLC',
          'disc_locally_protected', 'reputation_residual', 'reputation_converged')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9,
                     'axes.linewidth': .8, 'pdf.fonttype': 42,
                     'svg.fonttype': 'none', 'savefig.dpi': 300})


## Resident and rare-mutant equations

The auxiliary reputation flow starts from all-good reputations. Resident roots reached by relaxation are refined by step-limited Newton iterations to a residual of at most `POLISH_TOL`, so that classification is not limited by the relaxation tolerance. Points at which relaxation does not reach `RELAX_TOL` (slow convergence near folds) are resolved with the scalar self-consistency polynomial, selecting the highest attracting root, which is the root reached from all-good in the scalar flow. Classification uses the same rule as Extended Data Figure 5: ALLD is not excluded when $\lambda_{\rm ALLD}\ge-$`INVASION_TOL`; otherwise ALLC invades when $\lambda_{\rm ALLC}>$`INVASION_TOL`; otherwise DISC resists both. `RECOMPUTE_INVASION=True` recalculates the invasion grids (about 2.6 million resident solves).

In [2]:
def reputation_transform(phi, k1, k2, alpha=1.0, beta=1.0):
    phi, alpha, beta = np.broadcast_arrays(np.asarray(phi, float), np.asarray(alpha, float), np.asarray(beta, float))
    phi = np.clip(phi, 0.0, 1.0)
    one_minus = 1.0 - phi
    denominator = one_minus ** k1 + phi ** k2 - one_minus ** k1 * phi ** k2
    warning = np.zeros_like(phi)
    suspicious = np.zeros_like(phi)
    for m in range(1, k1 + 1):
        warning += (alpha * one_minus) ** m
    for n in range(1, k2 + 1):
        suspicious += phi ** n * (1.0 - beta ** n)
    numerator = phi ** (k2 + 1) * (1.0 + warning) + one_minus ** (k1 + 1) * suspicious
    return np.divide(numerator, denominator, out=np.zeros_like(numerator), where=np.abs(denominator) > 1e-15)

def effective_assessment(norm, e1, e2):
    p, q = NORM_PARAMETERS[norm]
    epsilon = (1.0 - e1) * (1.0 - e2) + e1 * e2
    p_cg = epsilon
    p_cb = p * (epsilon - e2) + q * (1.0 - epsilon - e2) + e2
    p_dg = e2
    p_db = q * (1.0 - 2.0 * e2) + e2
    return (p_cg, p_cb, p_dg, p_db)

def resident_rhs(g, norm, k1, k2, e1, e2, alpha, beta):
    p_cg, p_cb, p_dg, p_db = effective_assessment(norm, e1, e2)
    phi_disc = g ** 2 * p_cg + g * (1.0 - g) * (p_cb + p_dg) + (1.0 - g) ** 2 * p_db
    return reputation_transform(phi_disc, k1, k2, alpha, beta) - g

def solve_all_good_branch(norm, k1, k2, e1, e2, alpha, beta):
    e1, e2, alpha, beta = np.broadcast_arrays(np.asarray(e1, float), np.asarray(e2, float), np.asarray(alpha, float), np.asarray(beta, float))
    shape = alpha.shape
    e1f, e2f = (np.broadcast_to(e1, shape).ravel(), np.broadcast_to(e2, shape).ravel())
    af, bf = (alpha.ravel(), beta.ravel())
    g = np.ones_like(af)
    converged = np.zeros_like(g, dtype=bool)
    for _ in range(MAX_RELAX_STEPS):
        active = np.flatnonzero(~converged)
        if active.size == 0:
            break
        current = g[active]
        a1, a2, aa, bb = (e1f[active], e2f[active], af[active], bf[active])
        q1 = resident_rhs(current, norm, k1, k2, a1, a2, aa, bb)
        newly = np.abs(q1) <= RELAX_TOL
        converged[active[newly]] = True
        moving = active[~newly]
        if moving.size == 0:
            continue
        current = g[moving]
        a1, a2, aa, bb = (e1f[moving], e2f[moving], af[moving], bf[moving])
        q1 = q1[~newly]
        q2 = resident_rhs(np.clip(current + 0.5 * RELAX_DT * q1, 0, 1), norm, k1, k2, a1, a2, aa, bb)
        q3 = resident_rhs(np.clip(current + 0.5 * RELAX_DT * q2, 0, 1), norm, k1, k2, a1, a2, aa, bb)
        q4 = resident_rhs(np.clip(current + RELAX_DT * q3, 0, 1), norm, k1, k2, a1, a2, aa, bb)
        g[moving] = np.clip(current + RELAX_DT * (q1 + 2 * q2 + 2 * q3 + q4) / 6.0, 0, 1)
    residual = np.abs(resident_rhs(g, norm, k1, k2, e1f, e2f, af, bf))
    converged = residual <= RELAX_TOL
    return (g.reshape(shape), converged.reshape(shape), residual.reshape(shape))

def one_step_resident_root(norm, alpha, beta):
    """Select the highest stable scalar fixed point reached from all-good."""
    if MODEL['k1'] != 1 or MODEL['k2'] != 1:
        raise ValueError('The polynomial fallback is specific to one-step depths.')
    pcg, pcb, pdg, pdb = effective_assessment(norm, MODEL['e1'], MODEL['e2'])
    phi = Polynomial([pdb, pcb+pdg-2*pdb, pcg-pcb-pdg+pdb])
    numerator = (1-beta)*phi + (alpha+2*beta-1)*phi**2 + (1-alpha-beta)*phi**3
    denominator = 1-phi+phi**2
    polynomial = (numerator-Polynomial([0., 1.])*denominator).trim(1e-14)
    candidates = []
    for root in polynomial.roots():
        if abs(root.imag) > 1e-7 or not -1e-10 <= root.real <= 1+1e-10:
            continue
        r = float(np.clip(root.real, 0., 1.))
        for _ in range(10):
            derivative = polynomial.deriv()(r)
            if abs(derivative) < 1e-14:
                break
            step = polynomial(r)/derivative
            if not 0 <= r-step <= 1:
                break
            r -= step
        u = r*r*pcg + r*(1-r)*(pcb+pdg) + (1-r)**2*pdb
        residual = float(abs(reputation_transform(u, 1, 1, alpha, beta)-r))
        flow_derivative = float(polynomial.deriv()(r)/denominator(r))
        if residual <= RELAX_TOL and flow_derivative < -1e-12:
            candidates.append((r, residual, flow_derivative))
    if not candidates:
        raise RuntimeError(f'No residual-checked attracting root for {norm}, alpha={alpha}, beta={beta}.')
    return max(candidates)


def compute_invasion_fields(norm, alpha, beta):
    alpha, beta = np.broadcast_arrays(np.asarray(alpha, float), np.asarray(beta, float))
    r, converged, residual = solve_all_good_branch(norm, MODEL['k1'], MODEL['k2'],
        MODEL['e1'], MODEL['e2'], alpha, beta)
    if not converged.all():
        for index in np.flatnonzero(~converged.ravel()):
            fixed, error, _ = one_step_resident_root(norm, alpha.ravel()[index], beta.ravel()[index])
            r.ravel()[index] = fixed
            residual.ravel()[index] = error
            converged.ravel()[index] = True
    assert converged.all() and np.max(residual) <= RELAX_TOL
    r = polish_root(r, norm, alpha, beta)
    residual = np.abs(resident_rhs(r, norm, MODEL['k1'], MODEL['k2'], MODEL['e1'], MODEL['e2'], alpha, beta))
    assert residual.max() <= POLISH_TOL
    pcg, pcb, pdg, pdb = effective_assessment(norm, MODEL['e1'], MODEL['e2'])
    rc = reputation_transform(r*pcg + (1-r)*pcb, MODEL['k1'], MODEL['k2'], alpha, beta)
    rd = reputation_transform(r*pdg + (1-r)*pdb, MODEL['k1'], MODEL['k2'], alpha, beta)
    ratio = MODEL['b_over_c']
    ld = (1-MODEL['e1'])*(ratio*rd - (ratio-1)*r)
    lc = (1-MODEL['e1'])*(ratio*rc - 1 - (ratio-1)*r)
    return dict(r_ALLC=rc, r_ALLD=rd, r_DISC=r, lambda_ALLD=ld, lambda_ALLC=lc,
                disc_locally_protected=invasion_region(ld, lc) == 2,
                reputation_residual=residual, reputation_converged=converged)


def invasion_region(ld, lc):
    """0: ALLD not excluded; 1: ALLD excluded and ALLC invades; 2: DISC resists both (as in Extended Data Figure 5)."""
    return np.where(ld >= -INVASION_TOL, 0, np.where(lc > INVASION_TOL, 1, 2)).astype(np.int8)


def polish_root(r, norm, alpha, beta, iterations=40, h=1e-7):
    """Step-limited Newton refinement of resident roots (steps at most POLISH_STEP; one-sided differences at 0 and 1)."""
    r = np.array(r, float, copy=True)
    args = (norm, MODEL['k1'], MODEL['k2'], MODEL['e1'], MODEL['e2'], alpha, beta)
    for _ in range(iterations):
        value = resident_rhs(r, *args)
        if np.all(np.abs(value) <= 0.1 * POLISH_TOL):
            break
        lo, hi = np.clip(r - h, 0., 1.), np.clip(r + h, 0., 1.)
        slope = (resident_rhs(hi, *args) - resident_rhs(lo, *args)) / (hi - lo)
        step = np.where(np.abs(slope) > 1e-12, -value / np.where(slope == 0, 1., slope), 0.)
        r = np.clip(r + np.clip(step, -POLISH_STEP, POLISH_STEP), 0., 1.)
    return r


assert np.max(np.abs(reputation_transform(np.linspace(0, 1, 101), 1, 1, 0, 0)
                     - np.linspace(0, 1, 101))) < 1e-12


## Invasion grids and cooperation plane

The invasion grids are recomputed with `RECOMPUTE_INVASION=True` or loaded from `ESS_<norm>.npz`. The cooperation plane is loaded from the verified engine output; each row summarizes 512 shifted-Halton initial compositions (seed 2026) integrated to convergence. Cooperation is realized cooperation, including implementation errors. The earlier cooperation table (`previous_version/23_alpha_beta_dense41.csv`, computed with a resolution-23 interpolated reputation field and an unidentified generator) is retained only for comparison.

In [3]:
config_file = DATA / 'input_config.json'
cooperation_file = DATA / 'cooperation_readout_plane.csv'
if not config_file.is_file() or not cooperation_file.is_file():
    raise FileNotFoundError('Supply the data/extended_data_figure3_planes directory before running this notebook.')
config = json.loads(config_file.read_text())
if config['model'] != MODEL:
    raise ValueError('The saved cooperation configuration differs from MODEL. Use a separately validated cooperation scan for the new setting.')

cooperation = pd.read_csv(cooperation_file)
assert len(cooperation) == 4*41**2 and not cooperation.duplicated(['norm', 'alpha', 'beta']).any()
for key in ('b_over_c', 'e1', 'e2', 'k1', 'k2'):
    assert np.allclose(cooperation[key], MODEL[key])
assert cooperation['n_initial'].eq(512).all() and cooperation['outcome_gap_ok'].all()
print('Trajectories reaching the integration cap:', int(cooperation.n_step_cap.sum()), 'in', int((cooperation.n_step_cap > 0).sum()), 'conditions')
assert cooperation['mean_cooperation'].between(0, 1).all()

if RECOMPUTE_INVASION:
    values = np.linspace(0., 1., INVASION_POINTS)
    aa, bb = np.meshgrid(values, values, indexing='xy')
    fresh = {norm: compute_invasion_fields(norm, aa, bb) for norm in NORMS}
    for norm, fields in fresh.items():
        np.savez_compressed(DATA / f'ESS_{norm}.npz', alpha=values, beta=values, **fields)
    config.update(invasion_points=INVASION_POINTS,
                  invasion_source='Extended_Data_Figure3_Planes.ipynb (all-good relaxation, polynomial fallback, Newton refinement)',
                  invasion_relaxation=dict(dt=RELAX_DT, tolerance=RELAX_TOL, max_steps=MAX_RELAX_STEPS,
                                           polish_tolerance=POLISH_TOL, polish_max_step=POLISH_STEP))
    config.pop('invasion_source_sha256', None)
    config_file.write_text(json.dumps(config, indent=2)+'\n')

invasion = {}
for norm in NORMS:
    file = DATA / f'ESS_{norm}.npz'
    if not file.is_file():
        raise FileNotFoundError(f'Missing {file.name}; supply the saved fields or set RECOMPUTE_INVASION=True.')
    with np.load(file, allow_pickle=False) as saved:
        fields = {name: saved[name] for name in ('alpha', 'beta', *FIELDS)}
    expected = (config['invasion_points'], config['invasion_points'])
    assert fields['alpha'].shape == fields['beta'].shape == (expected[0],)
    for name in FIELDS:
        assert fields[name].shape == expected and np.isfinite(fields[name]).all()
    assert fields['reputation_converged'].all()
    assert fields['reputation_residual'].max() <= POLISH_TOL
    for name in ('r_ALLC', 'r_ALLD', 'r_DISC'):
        assert np.all((fields[name] >= 0) & (fields[name] <= 1))
    invasion[norm] = fields

classification = []
for norm in NORMS:
    field = invasion[norm]
    region = invasion_region(field['lambda_ALLD'], field['lambda_ALLC'])
    classification.append(dict(norm=norm, points=region.size,
        ALLD_not_excluded=int((region == 0).sum()), ALLC_only_invades=int((region == 1).sum()),
        both_excluded=int((region == 2).sum()), max_reputation_residual=float(field['reputation_residual'].max())))
pd.DataFrame(classification).to_csv(DATA / 'invasion_grid_summary.csv', index=False)
print('Loaded 6,724 cooperation rows and four complete, residual-checked invasion grids.')
display(pd.DataFrame(classification))


Trajectories reaching the integration cap: 26 in 7 conditions
Loaded 6,724 cooperation rows and four complete, residual-checked invasion grids.
  norm  points  ALLD_not_excluded  ALLC_only_invades  both_excluded  max_reputation_residual
0   SC  641601              44164             540728          56709             6.661338e-16
1   SH  641601             511289              88552          41760             5.551115e-16
2   SJ  641601             468897             126076          46628             6.106227e-16
3   SS  641601                  0             583537          58064             7.771561e-16


## Panels

Cooperation uses the original nonlinear color normalization. Invasion regions use the two invasion fitnesses with the tolerance stated above: ALLD non-exclusion has priority, then ALLC invasion, then resistance to both. This local three-strategy classification does not imply resistance to every strategy.

In [4]:
COOP_CMAP = LinearSegmentedColormap.from_list('publication_blue',
    ['#F8FBFD', '#EAF2F7', '#D4E4EE', '#B7D1E1', '#8FB7CF',
     '#659AB9', '#3F7CA4', '#215F8C', '#0F4670', '#082E4A'], N=256)
VALUE_KNOTS = np.array([0., .05, .15, .25, .40, .50, .55, .60, .65, .70, .75])
COLOR_KNOTS = np.array([0., .07, .15, .23, .32, .40, .48, .62, .78, .93, 1.])
COOP_NORM = FuncNorm((lambda x: np.interp(x, VALUE_KNOTS, COLOR_KNOTS),
                      lambda y: np.interp(y, COLOR_KNOTS, VALUE_KNOTS)), vmin=0., vmax=.75)
COOP_TICKS = [0., .15, .30, .45, .55, .60, .65, .70, .75]
REGION_COLORS = ['#D5A0A3', '#B3A4C7', '#80ADA5']
REGION_CMAP = ListedColormap(REGION_COLORS)
REGION_NORM = BoundaryNorm([-.5, .5, 1.5, 2.5], 3)
REGION_LABELS = ['ALLD not excluded', 'ALLD excluded; ALLC invades', 'DISC resists both']
LEGEND = [Patch(facecolor=color, edgecolor='none', label=label)
          for color, label in zip(REGION_COLORS, REGION_LABELS)]


def axes_style(ax, norm, panel=None):
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel=r'$\alpha$', ylabel=r'$\beta$', aspect='equal')
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.set_title(NORM_NAMES[norm], fontsize=10)
    if panel:
        ax.text(-.15, 1.05, panel, transform=ax.transAxes, fontweight='bold', fontsize=11)


def draw_cooperation(ax, norm, panel=None):
    part = cooperation.loc[cooperation.norm.eq(norm)]
    table = part.pivot(index='beta', columns='alpha', values='mean_cooperation').sort_index().sort_index(axis=1)
    assert table.shape == (41, 41) and np.isfinite(table).all().all()
    image = ax.pcolormesh(table.columns, table.index, table.to_numpy(), shading='nearest',
                         cmap=COOP_CMAP, norm=COOP_NORM, rasterized=True)
    axes_style(ax, norm, panel)
    return image


def draw_invasion(ax, norm, panel=None):
    field = invasion[norm]
    alpha, beta = field['alpha'], field['beta']
    ld, lc = field['lambda_ALLD'], field['lambda_ALLC']
    region = invasion_region(ld, lc)
    ax.pcolormesh(alpha, beta, region, shading='nearest', cmap=REGION_CMAP,
                  norm=REGION_NORM, rasterized=True)
    if ld.min() < 0 < ld.max():
        ax.contour(alpha, beta, ld, levels=[0], colors='white', linestyles='--', linewidths=1.2)
    if lc.min() < 0 < lc.max():
        ax.contour(alpha, beta, lc, levels=[0], colors='white', linestyles='-', linewidths=1.2)
    axes_style(ax, norm, panel)


def save_figure(fig, name):
    for extension in ('pdf', 'svg'):
        fig.savefig(FIGURES / f'{name}.{extension}', bbox_inches='tight', pad_inches=.04)


In [5]:
for col, norm in enumerate(NORMS):
    fig, ax = plt.subplots(figsize=(4.3, 3.7))
    image = draw_cooperation(ax, norm)
    fig.colorbar(image, ax=ax, fraction=.045, pad=.04, ticks=COOP_TICKS, label=r'Mean cooperation, $\bar\rho$')
    fig.tight_layout()
    save_figure(fig, f'cooperation_{norm}')
    plt.close(fig)

for col, norm in enumerate(NORMS):
    fig, ax = plt.subplots(figsize=(4.2, 4.4))
    fig.subplots_adjust(bottom=.27)
    draw_invasion(ax, norm)
    fig.legend(handles=LEGEND, loc='lower center', bbox_to_anchor=(.5, .01), frameon=False, fontsize=8)
    save_figure(fig, f'invasion_{norm}')
    plt.close(fig)

fig = plt.figure(figsize=(13.2, 7.2))
grid = fig.add_gridspec(2, 4, left=.065, right=.90, bottom=.16, top=.93, hspace=.36, wspace=.28)
for col, norm in enumerate(NORMS):
    ax = fig.add_subplot(grid[0, col])
    image = draw_cooperation(ax, norm, 'efgh'[col])
    if col:
        ax.set_ylabel('')
    ax = fig.add_subplot(grid[1, col])
    draw_invasion(ax, norm, 'ijkl'[col])
    if col:
        ax.set_ylabel('')
cax = fig.add_axes([.925, .58, .012, .33])
fig.colorbar(image, cax=cax, ticks=COOP_TICKS, label=r'Mean cooperation, $\bar\rho$')
fig.legend(handles=LEGEND, loc='lower center', bbox_to_anchor=(.49, .055), ncol=3,
           frameon=False, fontsize=8.5)
fig.text(.49, .015, 'White dashed: ALLD zero advantage; white solid: ALLC zero advantage.',
         ha='center', fontsize=8)
save_figure(fig, 'Extended_Data_Figure3_planes_combined')
plt.show()
plt.close(fig)
assert len(list(FIGURES.glob('*.pdf'))) == len(list(FIGURES.glob('*.svg'))) == 9
print('Saved eight panels and one combined figure in PDF and SVG.')


Saved eight panels and one combined figure in PDF and SVG.
